# Reconciliation of two 100-round DP-FedAvg runs

Compute **base, mints, wlsv, and ols** results from the completed 100-round training runs for GEF2017 and AusGrid2013.

- Use the training residuals saved by each 100-round model and reuse the original `FHF_Cloud` implementation with `ridge=1e-6`. No retraining or additional forecast or residual noise is applied.
- mints = `mint_shrinkage`, wlsv = `mint_var`, and ols = `mint_ols`.
- Test observations are used only for scoring. MASE / RMSSE retain the lag-1 training-target difference denominators, with metrics computed per series and then averaged arithmetically.
- Read parameters from each original privacy.json and verify 100 rounds and 100 aggregations.
- Input: `DP_fhf/outputs/01_dp_fedavg_100round/<dataset>/`.
- Output: **`DP_fhf/outputs/dp_100round_reconciliation/`**.

`training_epsilon` / `training_delta` record only the original DP-FedAvg training privacy parameters. They do not provide an end-to-end privacy guarantee for this reconciliation, which uses training residuals without additional privatization.

In [ ]:
# Locate the project package from a notebook folder or the project root.
from pathlib import Path
import os
import sys
_pphf_start = Path.cwd().resolve()
_pphf_override = os.environ.get("PPHF_PROJECT_ROOT") or os.environ.get("DP_FHF_PROJECT_ROOT")
_pphf_candidates = ([Path(_pphf_override).expanduser()] if _pphf_override else []) + [_pphf_start, *_pphf_start.parents]
PROJECT_ROOT = next((p.resolve() for p in _pphf_candidates if (p / "PPHF_modules" / "fhf" / "FHF_runner.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot locate PPHF_modules. Set PPHF_PROJECT_ROOT or start Jupyter inside the project.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pathlib import Path
import os
import sys
import importlib
import pandas as pd
from IPython.display import display

# Start from any project subdirectory, or specify the project root with an environment variable.
candidates = []
if os.environ.get("DP_FHF_PROJECT_ROOT"):
    candidates.append(Path(os.environ["DP_FHF_PROJECT_ROOT"]).expanduser())
candidates.extend([Path.cwd(), *Path.cwd().parents])
ROOT = next((p.resolve() for p in candidates
             if (p / "PPHF_modules" / "fhf").is_dir() and (p / "DP_fhf").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Set DP_FHF_PROJECT_ROOT to the FHF_NDSH project directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from PPHF_modules.dp_fhf import reconciliation_100round
importlib.reload(reconciliation_100round)

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda value: f"{value:.6f}")
print("Project:", ROOT)
print("Python:", sys.executable)

In [ ]:
DATASETS = ("GEF2017", "AusGrid2013")
OUTPUT_ROOT = ROOT / "DP_fhf" / "outputs" / "dp_100round_reconciliation"
RIDGE = 1e-6
SAVE_FORECASTS = True
print("Output:", OUTPUT_ROOT)

## Compute and save

Recompute both runs and verify the original base metrics, matrix projections, and hierarchical aggregation consistency. Replace the output directory only after the complete batch succeeds.

In [ ]:
summary = reconciliation_100round.run_100round_reconciliation(
    root=ROOT, datasets=DATASETS, output_root=OUTPUT_ROOT,
    ridge=RIDGE, save_forecasts=SAVE_FORECASTS,
)
assert len(summary) == 4 * len(DATASETS)
display(summary[["dataset", "num_rounds", "noise_multiplier", "clip_norm", "method", "MAE", "RMSE", "MASE", "RMSSE"]])

## Side-by-side MASE / RMSSE comparison

In [ ]:
comparison = summary.pivot(index="dataset", columns="method", values=["MASE", "RMSSE"])
comparison = comparison.reindex(columns=pd.MultiIndex.from_product(
    [["MASE", "RMSSE"], ["base", "mints", "wlsv", "ols"]], names=comparison.columns.names))
display(comparison)
print("Relative change vs base; negative means lower error:")
display(summary.loc[summary.method != "base", ["dataset", "method", "change_MASE_pct_vs_base", "change_RMSSE_pct_vs_base"]])

## Files and validation

The output root contains `metrics_summary.csv` (8 rows), `metrics_wide.csv` (2 rows), per-series metrics, and matrix checks.
Each dataset folder contains pointwise predictions in `reconciled_forecasts.csv.gz`, S/G/P/W matrices, run metrics, and input/code SHA256 hashes.
Compressed predictions can be read directly with `pd.read_csv`.

In [ ]:
import json
manifest = json.loads((OUTPUT_ROOT / "manifest.json").read_text())
checks = pd.read_csv(OUTPUT_ROOT / "matrix_diagnostics.csv")
assert manifest["status"] == "complete"
assert manifest["runs_completed"] == len(DATASETS)
assert checks.PS_minus_S_max_abs.max() < 1e-7
assert checks.coherence_relative.max() < 1e-9
assert all(run["source_files_unchanged"] for run in manifest["run_manifests"])
print("Maximum |PS-S|:", checks.PS_minus_S_max_abs.max())
print("Maximum relative coherence error:", checks.coherence_relative.max())
print("Summary:", OUTPUT_ROOT / "metrics_summary.csv")
print("Wide comparison:", OUTPUT_ROOT / "metrics_wide.csv")